# Clean and Process df

## Set Up

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH, SEED
from src.utils import col_config_og, col_config_sensitivity
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 500)

In [ ]:
df = pd.read_csv(
    BASE_PATH / "data" / "complete_raw" / "comppsych_dawbas_20210414-0.csv"
)
df.shape

In [ ]:
self_impact_cols = col_config_og["self_impact"]
parent_impact_cols = col_config_og["parent_impact"]
self_text_cols = col_config_og["self_text"]
parent_text_cols = col_config_og["parent_text"]
self_quest_cols = col_config_og["self_quest"]
parent_quest_cols = col_config_og["parent_quest"]
meta_cols = col_config_og["meta"]

## Clean

Remove duplicated SDAN (23717.0) at idx 234 (double the amount of null), same patient

In [ ]:
df_clean = df.drop(index=234)

Remove participants who `removed consent`

In [ ]:
df_clean = df_clean[df_clean["Eligible"] != "Did not or withdrew assent/consent"].copy()

Create new eligibility col to stratify on excluded bc meets criteria, excluded bc not meet criteria, & everything else

In [ ]:
mapping = {
    "Excluded: meets exclusionary criteria (substance use, psychosis, etc.)": "excluded",
    "Excluded: does not meet criteria": "excluded",
    "Excluded: cannot be reached or scheduled, all contact options exhausted": "Include",
    "Include: can't scan (braces, etc.)": "Include",
}

df_clean["eligible_group"] = df_clean["Eligible"].map(mapping).fillna("Include")
df_clean["eligible_group"].value_counts()

Fill participant type NAs

In [ ]:
df_clean["Participant_Type2"] = df_clean["Participant_Type2"].fillna("UNSURE")

Subset to only include relevant depression + general cols

In [ ]:
all_self_cols = self_quest_cols + self_text_cols + self_impact_cols
all_parent_cols = parent_quest_cols + parent_text_cols + parent_impact_cols
all_depr_cols = all_self_cols + all_parent_cols
keep_cols = meta_cols + all_depr_cols
df_subset = df_clean[keep_cols]
df_subset.shape

Split

In [ ]:
from sklearn.model_selection import train_test_split
from shutil import rmtree

##### export raw
export_dir = BASE_PATH / "data" / "cleaned" / "initial_split"
if export_dir.exists():
    rmtree(export_dir)
export_dir.mkdir(exist_ok=False, parents=True)
df_subset.to_parquet(export_dir / "raw_clean_depression.parquet")

##### Split
train, test, _, _ = train_test_split(
    df_subset,
    np.ones(len(df_subset)),
    train_size=0.67,
    random_state=SEED,
    shuffle=True,
    stratify=df_subset["eligible_group"],
)
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)
## Export
train.to_parquet(export_dir / "train.parquet")
test.to_parquet(export_dir / "test.parquet")

## Work only on train set

In [ ]:
df_sub = pd.read_parquet(
    BASE_PATH / "data" / "cleaned" / "initial_split" / "train.parquet"
)
df_sub.shape

Seperate self & parent dfs

In [ ]:
self_df = df_sub[meta_cols + self_quest_cols + self_text_cols + self_impact_cols].copy()
parent_df = df_sub[
    meta_cols + parent_quest_cols + parent_text_cols + parent_impact_cols
].copy()

## Fill/drop NAs


- Create cols for Depression. Irritability, Loss of Interest checks
- Used to skip questions later
- Skipped questions due to these^ will be marked as 0 (questionairre) or "Not Applicable" (text)
- Non-structural NAs should be dropped
- Fill text for participants only missing 1/2 entries

### Self

Create cols

In [ ]:
## Impact score
self_df["impact_score_self"] = self_df[self_impact_cols].sum(axis=1)
# Depr --> H1, H2, AND H3
self_df["Depression_Check_self"] = (
    (self_df["s_dep_1_sad"] == 1)
    & (self_df["s_dep_2_miserable_daily"] == 1)
    & (self_df["s_dep_3_miserable_mostly"] == 1)
)
# Irr --> H7, H8, AND H9
self_df["Irritable_Check_self"] = (
    (self_df["s_dep_7_irritable"] == 1)
    & (self_df["s_dep_8_irritable_daily"] == 1)
    & (self_df["s_dep_9_irratable_mostly"] == 1)
)
# Loss of Interest --> H13 AND H14
self_df["Loss_Interest_Check_self"] = (self_df["s_dep_13_anhedonia"] == 1) & (
    self_df["s_dep_14_anhedonia_daily"] == 1
)

Mark NA as 0 for skipped questions (bc of answer to a previous question)

In [ ]:
## ====> Miserable (2-5); For 2-5, change NAs to 0s when H1 is 0
skipped_after_1 = [
    "s_dep_2_miserable_daily",
    "s_dep_3_miserable_mostly",
    "s_dep_4_miserable_cheered_up",
    "s_dep_5_miserable_duration",
]
mask = self_df["s_dep_1_sad"] == 0
self_df.loc[mask, skipped_after_1] = self_df.loc[mask, skipped_after_1].fillna(0)
## ====> Irritiable (8-11); For 8-11, change NAs to 0s when H7 is 0
skipped_after_7 = [
    "s_dep_8_irritable_daily",
    "s_dep_9_irratable_mostly",
    "s_dep_10_irritable_improved",
    "s_dep_11_irritable_duration",
]
mask = self_df["s_dep_7_irritable"] == 0
self_df.loc[mask, skipped_after_7] = self_df.loc[mask, skipped_after_7].fillna(0)
## ====> anhedonia (14-16); for 14-16, change NAs to 0s when H13 is 0
skipped_after_13 = [
    "s_dep_14_anhedonia_daily",
    "s_dep_15_anhedonia_mostly",
    "s_dep_16_anhedonia_duration",
]
mask = self_df["s_dep_13_anhedonia"] == 0
self_df.loc[mask, skipped_after_13] = self_df.loc[mask, skipped_after_13].fillna(0)
## ====> H17; Answer only if H13 AND (Depr OR Irr); Change NA to 0 if conditition not met
mask = (self_df["s_dep_13_anhedonia"] == 0) | (
    (self_df["Depression_Check_self"] == 0) & (self_df["Irritable_Check_self"] == 0)
)
self_df.loc[mask, "s_dep_17_coincided_mis_irrit"] = self_df.loc[
    mask, "s_dep_17_coincided_mis_irrit"
].fillna(0)
## ====> Depr + Irr + LoI (18A-18L, 19, 20A-20D, 21)
# Only need 1 to be True to answer
skipped_at_18 = [
    "s_dep_18a_tired",
    "s_dep_18b_appetite",
    "s_dep_18c_weight",
    "s_dep_18d_insomnia",
    "s_dep_18e_hypersomnia",
    "s_dep_18f_agitation",
    "s_dep_18g_worthless",
    "s_dep_18h_concentration",
    "s_dep_18i_thoughts_death",
    "s_dep_18j_self_harm_talk_recent",
    "s_dep_18k_self_harm_act_recent",
    "s_dep_18l_self_harm_act_ever",
] + self_impact_cols
# Fill questions with 0
self_df.loc[mask, skipped_at_18] = self_df.loc[mask, skipped_at_18].fillna(0)

Text questions only answered if one of Depr/Irr/LoI

- Remove participants w/o text

In [ ]:
self_df["impact_score_self"] = self_df[self_impact_cols].sum(axis=1)
mask = (
    (self_df["Depression_Check_self"] == 0)
    & (self_df["Irritable_Check_self"] == 0)
    & (self_df["Loss_Interest_Check_self"] == 0)
)
## remove participants without text
self_df_filter = self_df[~mask]

Remove death/self-harm questions bc of missingness (>40 in each of 18i, 18j, 18k,16l)

In [ ]:
drop_cols = [
    "s_dep_18i_thoughts_death",
    "s_dep_18j_self_harm_talk_recent",
    "s_dep_18k_self_harm_act_recent",
    "s_dep_18l_self_harm_act_ever",
]
self_df_sub = self_df_filter.drop(drop_cols, axis=1)

Remove NAs resulting from true missingness rather than structure
- All have missing for text cols

In [ ]:
## From questionnaire cols
og_len = len(self_df_sub)
self_quest_cols = col_config_sensitivity["self_quest"]
self_df_sub = self_df_sub.dropna(subset=self_quest_cols)
new_len = len(self_df_sub)
print(f"Total dropped bc questionnaire NAs: {og_len - new_len}")
## From text cols
og_len = len(self_df_sub)
self_df_sub = self_df_sub.dropna(subset=self_text_cols, thresh=5)
new_len = len(self_df_sub)
print(f"Total dropped bc text NAs: {og_len - new_len}")

Instead of removing entire participants, fill with "Not answered"
- Two participants only missing 2 text entries, fill with "Not answered" (DAWBA_IDs 234991, 234934)

- Two participants w/ only 1 missing entry (IDs 234884, 234474)

In [ ]:
self_df_sub[self_text_cols] = self_df_sub[self_text_cols].fillna("Not answered")

In [ ]:
assert self_df_sub.isna().sum().sum() == 0

In [ ]:
print(self_df["impact_score_self"].var())
print(self_df_sub["impact_score_self"].var())

### Parent

Create cols

In [ ]:
# Depr --> H1, H2, AND H3
parent_df["Depression_Check_parent"] = (
    (parent_df["p_dep_1_sad"] == 1)
    & (parent_df["p_dep_2_miserable_daily"] == 1)
    & (parent_df["p_dep_3_miserable_mostly"] == 1)
)
# Irr --> H7, H8, AND H9
parent_df["Irritable_Check_parent"] = (
    (parent_df["p_dep_7_irritable"] == 1)
    & (parent_df["p_dep_8_irritable_daily"] == 1)
    & (parent_df["p_dep_9_irratable_mostly"] == 1)
)
# Loss of Interest --> H13 AND H14
parent_df["Loss_Interest_Check_parent"] = (parent_df["p_dep_13_anhedonia"] == 1) & (
    parent_df["p_dep_14_anhedonia_daily"] == 1
)

Mark NA as 0 for skipped questions

In [ ]:
## ====> Miserable (2-5); For 2-5, change NAs to 0s when H1 is 0
skipped_after_1 = [
    "p_dep_2_miserable_daily",
    "p_dep_3_miserable_mostly",
    "p_dep_4_miserable_cheered_up",
    "p_dep_5_miserable_duration",
]
mask = parent_df["p_dep_1_sad"] == 0
parent_df.loc[mask, skipped_after_1] = parent_df.loc[mask, skipped_after_1].fillna(0)
## ====> Irritiable (8-11); For 8-11, change NAs to 0s when H7 is 0
skipped_after_7 = [
    "p_dep_8_irritable_daily",
    "p_dep_9_irratable_mostly",
    "p_dep_10_irritable_improved",
    "p_dep_11_irritable_duration",
]
mask = parent_df["p_dep_7_irritable"] == 0
parent_df.loc[mask, skipped_after_7] = parent_df.loc[mask, skipped_after_7].fillna(0)
## ====> anhedonia (14-16); for 14-16, change NAs to 0s when H13 is 0
skipped_after_13 = [
    "p_dep_14_anhedonia_daily",
    "p_dep_15_anhedonia_mostly",
    "p_dep_16_anhedonia_duration",
]
mask = parent_df["p_dep_13_anhedonia"] == 0
parent_df.loc[mask, skipped_after_13] = parent_df.loc[mask, skipped_after_13].fillna(0)
## ====> H17; Answer only if H13 AND (Depr OR Irr); Change NA to 0 if conditition not met
mask = (parent_df["p_dep_13_anhedonia"] == 0) | (
    (parent_df["Depression_Check_parent"] == 0)
    & (parent_df["Irritable_Check_parent"] == 0)
)
parent_df.loc[mask, "p_dep_17_coincided_mis_irrit"] = parent_df.loc[
    mask, "p_dep_17_coincided_mis_irrit"
].fillna(0)
## ====> Depr + Irr + LoI (18A-18L, 19, 20A-20D, 21)
# Only need 1 to be True to answer
skipped_at_18 = [
    "p_dep_18a_tired",
    "p_dep_18b_appetite",
    "p_dep_18c_weight",
    "p_dep_18d_insomnia",
    "p_dep_18e_hypersomnia",
    "p_dep_18f_agitation",
    "p_dep_18g_worthless",
    "p_dep_18h_concentration",
    "p_dep_18i_thoughts_death",
    "p_dep_18j_self_harm_talk_recent",
    "p_dep_18k_self_harm_act_recent",
    "p_dep_18l_self_harm_act_ever",
] + parent_impact_cols
# Fill questions with 0
parent_df.loc[mask, skipped_at_18] = parent_df.loc[mask, skipped_at_18].fillna(0)

Text questions only answered if one of Depr/Irr/LoI

- Remove participants w/o text

In [ ]:
parent_df["impact_score_parent"] = parent_df[parent_impact_cols].sum(axis=1)
mask = (
    (parent_df["Depression_Check_parent"] == 0)
    & (parent_df["Irritable_Check_parent"] == 0)
    & (parent_df["Loss_Interest_Check_parent"] == 0)
)
## remove participants without text
parent_df_filter = parent_df[~mask]
other = parent_df[mask]

print(parent_df["impact_score_parent"].var())
print(parent_df_filter["impact_score_parent"].var())

Remove death/self-harm questions bc of missingness (40 in each of 18i, 18j, 18k,16l)

In [ ]:
drop_cols = [
    "p_dep_18i_thoughts_death",
    "p_dep_18j_self_harm_talk_recent",
    "p_dep_18k_self_harm_act_recent",
    "p_dep_18l_self_harm_act_ever",
]
parent_df_sub = parent_df_filter.drop(drop_cols, axis=1)

Remove NAs resulting from true missingness rather than structure
- All have missing for text cols

In [ ]:
## From questionnaire cols
og_len = len(parent_df_sub)
parent_quest_cols = col_config_sensitivity["parent_quest"]
parent_df_sub = parent_df_sub.dropna(subset=parent_quest_cols)
new_len = len(parent_df_sub)
print(f"Total dropped bc questionnaire NAs: {og_len - new_len}")
## From text cols
og_len = len(parent_df_sub)
parent_df_sub = parent_df_sub.dropna(subset=parent_text_cols, thresh=5)
new_len = len(parent_df_sub)
print(f"Total dropped bc text NAs: {og_len - new_len}")

Instead of removing entire participants, fill with "Not answered"

- One participants w/ only 1 missing entry (IDs 234934)

In [ ]:
parent_df_sub[parent_text_cols] = parent_df_sub[parent_text_cols].fillna("Not answered")

In [ ]:
assert parent_df_sub.isna().sum().sum() == 0

In [ ]:
print(parent_df["impact_score_parent"].var())
print(parent_df_sub["impact_score_parent"].var())

In [ ]:
import matplotlib.pyplot as plt

plt.hist(parent_df["impact_score_parent"])
plt.title("Parent")

## Export

Export base dfs

In [ ]:
export_dir = BASE_PATH / "data" / "cleaned" / "processed"
if export_dir.exists():
    rmtree(export_dir)
export_dir.mkdir(exist_ok=False, parents=True)
# Self
self_df_sub = self_df_sub.reset_index(drop=True)
self_df_sub.to_parquet(export_dir / "self.parquet")
# Parent
parent_df_sub = parent_df_sub.reset_index(drop=True)
parent_df_sub.to_parquet(export_dir / "parent.parquet")

# Also process meta info file for easier use later

In [ ]:
from shutil import rmtree

self_text_cols = col_config_sensitivity["self_text"]
parent_text_cols = col_config_sensitivity["parent_text"]
meta_df = pd.read_excel(
    BASE_PATH
    / "data"
    / "complete_raw"
    / "dawba_column_names_and_descriptions_edit.xlsx"
)

In [ ]:
keep_cols = ["new_name", "description"]
self_meta = meta_df[meta_df["new_name"].isin(self_text_cols)][keep_cols]
parent_meta = meta_df[meta_df["new_name"].isin(parent_text_cols)][keep_cols]
## Ensure we got exactly right number of columns
assert len(self_meta) == len(self_text_cols)
assert len(parent_meta) == len(parent_text_cols)

In [ ]:
def strip_prefix(meta_df):
    new_rows_list = []
    for idx, row in meta_df.iterrows():

        og_text = row.description
        og_text_split = og_text.split(")")
        # Get only text after `Dep: text n)` prefix + remove leading/trailing whitespace + remove new lines + remove double spaces
        new_text = (
            " ".join(og_text_split[1:]).strip().replace("\n", " ").replace("  ", " ")
        )
        cur_row = {"question_col": row.new_name, "question_text": new_text}
        new_rows_list.append(cur_row)
    return pd.DataFrame(new_rows_list)

In [ ]:
self_meta_cleaned = strip_prefix(self_meta)
parent_meta_cleaned = strip_prefix(parent_meta)

Export

In [ ]:
meta_export_dir = BASE_PATH / "data" / "cleaned" / "meta_text"
if meta_export_dir.exists():
    rmtree(meta_export_dir)
meta_export_dir.mkdir(exist_ok=False, parents=True)
self_meta_cleaned.to_csv(meta_export_dir / "self.csv")
parent_meta_cleaned.to_csv(meta_export_dir / "parent.csv")